In [5]:
!pip install -q transformers peft datasets accelerate scikit-learn matplotlib seaborn

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
import os
import time
import json
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import matthews_corrcoef, accuracy_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, AdaLoraConfig, get_peft_model, TaskType
from dataclasses import dataclass, field
from typing import Optional, List, Dict

DATA_DIR = "/kaggle/input/datasets/ak236789/cola-glue-data2" 

OUTPUT_DIR = "/kaggle/working/outputs"
PLOTS_DIR = "/kaggle/working/plots"
MODEL_NAME = "microsoft/deberta-v3-base"
NUM_LABELS = 2

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

@dataclass
class TrainingConfig:
    seed: int = 42
    max_seq_length: int = 128
    num_epochs: int = 10
    batch_size: int = 32
    learning_rate: float = 2e-4       
    backbone_lr: float = 0.0
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1          
    max_grad_norm: float = 1.0
    
    # LoRA common
    lora_rank: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    target_modules: list = field(default_factory=lambda: [
        "query_proj", "key_proj", "value_proj"   
    ])
    
    # AdaLoRA specific
    adalora_init_rank: int = 12
    adalora_target_rank: int = 4
    adalora_tinit: int = 200
    adalora_tfinal: int = 1000
    adalora_delta_t: int = 10
    
    # SoRA specific
    sora_rank: int = 8
    sora_lambda: float = 0.5
    sora_proximal_lr: float = 1e-2     
    
    # Evaluation
    eval_steps: int = 50
    save_steps: int = 200
    logging_steps: int = 10

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = TrainingConfig()
set_seed(config.seed)
print(f"Device: {device}")

Device: cuda


In [4]:
import csv

class CoLADataset(Dataset):
    def __init__(self, split: str, tokenizer, max_length: int = 128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.sentences = []
        self.labels = []
        
        filepath = os.path.join(DATA_DIR, f"{split}.tsv")
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.reader(f, delimiter='\t')
            for row in reader:
                if len(row) >= 4:
                    self.sentences.append(row[3])
                    self.labels.append(int(row[1]))
                elif len(row) == 2 and split == 'test':
                    self.sentences.append(row[1])
                    self.labels.append(0)
        
        print(f"Loaded {len(self.sentences)} examples from CoLA {split} set")
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.sentences[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_dataset = CoLADataset('train', tokenizer, config.max_seq_length)
val_dataset = CoLADataset('dev', tokenizer, config.max_seq_length)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Loaded 8551 examples from CoLA train set
Loaded 1043 examples from CoLA dev set
Train batches: 268, Val batches: 33


In [5]:
from transformers import get_linear_schedule_with_warmup

def compute_metrics(predictions, labels):
    preds = np.argmax(predictions, axis=1)
    mcc = matthews_corrcoef(labels, preds)
    acc = accuracy_score(labels, preds)
    return {'mcc': mcc, 'accuracy': acc}

def count_parameters(model, method):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        'method': method, 'total_params': total,
        'trainable_params': trainable, 'frozen_params': total - trainable,
        'trainable_percent': 100.0 * trainable / total if total > 0 else 0,
    }

def compute_effective_rank(model, method):
    if method == 'sora' and hasattr(model, 'get_effective_ranks'):
        return model.get_effective_ranks()
    ranks = {}
    for name, module in model.named_modules():
        if hasattr(module, 'lora_A') and hasattr(module, 'lora_B'):
            A = module.lora_A.data if isinstance(module.lora_A, nn.Parameter) else module.lora_A.default.weight.data
            B = module.lora_B.data if isinstance(module.lora_B, nn.Parameter) else module.lora_B.default.weight.data
            W = B @ A
            S = torch.linalg.svdvals(W.float())
            S_norm = S / S.sum().clamp(min=1e-9)
            entropy = -(S_norm * torch.log(S_norm + 1e-9)).sum()
            ranks[name] = {'effective_rank': torch.exp(entropy).item(), 'max_rank': min(W.shape)}
    return ranks

def evaluate(model, dataloader, device, is_sora=False):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, num_batches = 0.0, 0
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=batch['labels'])
            total_loss += outputs.loss.item()
            all_preds.append(outputs.logits.cpu().numpy())
            all_labels.append(batch['labels'].cpu().numpy())
            num_batches += 1
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    metrics = compute_metrics(all_preds, all_labels)
    metrics['loss'] = total_loss / max(num_batches, 1)
    return metrics

print("✓ Utility functions defined")

✓ Utility functions defined


In [6]:
class SoRALinear(nn.Module):
    """SoRA layer: ΔW = B · diag(g) · A"""
    def __init__(self, in_features, out_features, rank=8, alpha=16.0, dropout=0.1):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.empty(out_features, rank))
        self.gate = nn.Parameter(torch.ones(rank))
        self.lora_dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
        nn.init.ones_(self.gate)

    def forward(self, x):
        x_dropped = self.lora_dropout(x)
        down = F.linear(x_dropped, self.lora_A)
        gated = down * self.gate
        up = F.linear(gated, self.lora_B)
        return up * self.scaling

    def get_effective_rank(self, threshold=1e-5):
        return (self.gate.abs() > threshold).sum().item()

    def get_gate_stats(self):
        g = self.gate.detach()
        return {
            'effective_rank': self.get_effective_rank(),
            'max_rank': self.rank,
            'gate_mean': g.abs().mean().item(),
            'gate_std': g.std().item(),
            'gate_min': g.abs().min().item(),
            'gate_max': g.abs().max().item(),
            'gate_zeros': (g.abs() < 1e-5).sum().item(),
            'gate_sparsity': (g.abs() < 1e-5).sum().item() / self.rank,
        }


class SoRAModel(nn.Module):
    """Wraps a pretrained model with SoRA adapters."""
    def __init__(self, base_model, target_modules, rank=8, alpha=16.0, dropout=0.1):
        super().__init__()
        self.base_model = base_model
        self.sora_layers: Dict[str, SoRALinear] = {}
        for param in self.base_model.parameters():
            param.requires_grad = False
        self._inject_sora_adapters(target_modules, rank, alpha, dropout)
    def _inject_sora_adapters(self, target_modules, rank, alpha, dropout):
        for name, module in self.base_model.named_modules():
            if any(target in name for target in target_modules):
                if isinstance(module, nn.Linear):
                    sora_layer = SoRALinear(
                        module.in_features, module.out_features,
                        rank, alpha, dropout
                    )
                    layer_key = name.replace('.', '_')
                    self.sora_layers[layer_key] = sora_layer
                    self.add_module(f"sora_{layer_key}", sora_layer)
                    self._hook_module(module, sora_layer)
        print(f"Injected SoRA adapters into {len(self.sora_layers)} layers")

    def _hook_module(self, original_module, sora_layer):
        original_forward = original_module.forward

        def hooked_forward(x):
            return original_forward(x) + sora_layer(x)

        original_module.forward = hooked_forward

    def forward(self, **kwargs):
        return self.base_model(**kwargs)
    def get_sora_parameters(self):
        """A and B matrices only — no gates."""
        return [p for n, p in self.named_parameters() if 'sora_' in n and 'gate' not in n]
    def get_gate_parameters(self):
        """Gate vectors only."""
        return [p for n, p in self.named_parameters() if 'gate' in n]

    def count_trainable_parameters(self):
        sora_params = sum(p.numel() for p in self.get_sora_parameters())
        gate_params = sum(p.numel() for p in self.get_gate_parameters())
        total_params = sum(p.numel() for p in self.base_model.parameters())
        trainable_params = sora_params + gate_params
        return {
            'total_params': total_params,
            'trainable_params': trainable_params,
            'sora_ab_params': sora_params,
            'gate_params': gate_params,
            'trainable_percent': 100.0 * trainable_params / total_params,
        }
    def get_effective_ranks(self):
        return {key: layer.get_gate_stats() for key, layer in self.sora_layers.items()}
    def get_avg_effective_rank(self):
        ranks = [layer.get_effective_rank() for layer in self.sora_layers.values()]
        return sum(ranks) / len(ranks) if ranks else 0.0


class ProximalSGD:

    def __init__(self, gate_params, lr=1e-2, lam=0.5):
        self.gate_params = list(gate_params)   # materialise iterator once
        self.lr = lr
        self.lam = lam

    @staticmethod
    def soft_threshold(x, threshold):
        return torch.sign(x) * torch.clamp(torch.abs(x) - threshold, min=0.0)

    def step(self):
        threshold = self.lr * self.lam
        for param in self.gate_params:
            with torch.no_grad():
                # Gradient descent step (only if gradient exists)
                if param.grad is not None:
                    param.data.sub_(self.lr * param.grad)
                # Proximal operator — always applied to enforce sparsity
                param.data = self.soft_threshold(param.data, threshold)

    def zero_grad(self):
        for param in self.gate_params:
            if param.grad is not None:
                param.grad.zero_()


def compute_sora_l1_loss(model: SoRAModel) -> torch.Tensor:
    device = next(model.parameters()).device
    l1_loss = torch.tensor(0.0, device=device)
    for layer in model.sora_layers.values():
        l1_loss = l1_loss + layer.gate.abs().sum()
    return l1_loss

In [7]:
set_seed(config.seed)
print("Training LoRA")

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=8, alpha=16, dropout=0.1):
        super().__init__()
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))  # ← SAME AS SORA
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return (self.dropout(x) @ self.lora_A.T @ self.lora_B.T) * self.scaling

class LoRAModel(nn.Module):
    def __init__(self, base_model, target_modules, rank=8, alpha=16, dropout=0.1):
        super().__init__()
        self.base_model = base_model
        self.lora_layers = nn.ModuleDict()
        for p in self.base_model.parameters():
            p.requires_grad = False
        for name, module in self.base_model.named_modules():
            if any(t in name for t in target_modules) and isinstance(module, nn.Linear):
                key = name.replace('.', '_')
                lora = LoRALinear(module.in_features, module.out_features, rank, alpha, dropout)
                self.lora_layers[key] = lora
                orig = module.forward
                module.forward = self._make_hook(orig, lora)
        for n, p in self.base_model.named_parameters():
            if 'classifier' in n or 'pooler' in n:
                p.requires_grad = True
        t = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"Injected LoRA into {len(self.lora_layers)} layers")
        print(f"trainable params: {t:,} || all params: {total:,} || trainable%: {100*t/total:.4f}")

    @staticmethod
    def _make_hook(orig_fn, lora_fn):
        def hooked(x):
            return orig_fn(x) + lora_fn(x)
        return hooked

    def forward(self, **kwargs):
        return self.base_model(**kwargs)

base_model_lora = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).float()
model_lora = LoRAModel(base_model_lora, ["query_proj","key_proj","value_proj"],
    rank=config.lora_rank, alpha=config.lora_alpha, dropout=config.lora_dropout).to(device)

lora_params = [p for n,p in model_lora.named_parameters() if 'lora_' in n and p.requires_grad]
head_params = [p for n,p in model_lora.named_parameters() if ('classifier' in n or 'pooler' in n) and p.requires_grad]

optimizer_lora = AdamW([
    {'params': lora_params, 'lr': 3e-4},
    {'params': head_params, 'lr': 2e-3},
], weight_decay=config.weight_decay)

NUM_EPOCHS_LORA = 20
total_steps = len(train_loader) * NUM_EPOCHS_LORA
scheduler_lora = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_lora, max_lr=config.learning_rate,
    total_steps=total_steps, pct_start=config.warmup_ratio, anneal_strategy='cos')

lora_results = {'method':'lora','train_losses':[],'val_losses':[],'val_mccs':[],'val_accs':[],'epoch_times':[],'best_mcc':-1.0,'best_epoch':0}
start_time = time.time()

for epoch in range(NUM_EPOCHS_LORA):
    t0 = time.time()
    model_lora.train()
    eloss, nb = 0.0, 0
    for batch in train_loader:
        batch = {k:v.to(device) for k,v in batch.items()}
        out = model_lora(input_ids=batch['input_ids'],attention_mask=batch['attention_mask'],labels=batch['labels'])
        if torch.isnan(out.loss): optimizer_lora.zero_grad(); continue
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model_lora.parameters() if p.requires_grad], 1.0)
        optimizer_lora.step(); scheduler_lora.step(); optimizer_lora.zero_grad()
        eloss += out.loss.item(); nb += 1
    val = evaluate(model_lora, val_loader, device)
    lora_results['train_losses'].append(eloss/max(nb,1))
    lora_results['val_losses'].append(val['loss'])
    lora_results['val_mccs'].append(val['mcc'])
    lora_results['val_accs'].append(val['accuracy'])
    lora_results['epoch_times'].append(time.time()-t0)
    if val['mcc']>lora_results['best_mcc']:
        lora_results['best_mcc']=val['mcc']; lora_results['best_epoch']=epoch+1
    print(f"Epoch {epoch+1}/{NUM_EPOCHS_LORA} | {time.time()-t0:.0f}s | Loss: {eloss/max(nb,1):.4f} | MCC: {val['mcc']:.4f} | Acc: {val['accuracy']:.4f}")

lora_results['total_time'] = time.time()-start_time
lora_results['effective_ranks'] = compute_effective_rank(model_lora, 'lora')
lora_results['param_counts'] = count_parameters(model_lora, 'lora')
print(f"\n✓ LoRA Best MCC: {lora_results['best_mcc']:.4f} (epoch {lora_results['best_epoch']})")

Training LoRA


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

Injected LoRA into 36 layers
trainable params: 1,034,498 || all params: 184,866,050 || trainable%: 0.5596
Epoch 1/20 | 179s | Loss: 0.6186 | MCC: 0.0000 | Acc: 0.6913
Epoch 2/20 | 186s | Loss: 0.6144 | MCC: 0.0000 | Acc: 0.6913
Epoch 3/20 | 186s | Loss: 0.6083 | MCC: 0.0000 | Acc: 0.6913
Epoch 4/20 | 185s | Loss: 0.6028 | MCC: 0.0000 | Acc: 0.6913
Epoch 5/20 | 185s | Loss: 0.5882 | MCC: 0.0976 | Acc: 0.6961
Epoch 6/20 | 186s | Loss: 0.5707 | MCC: 0.0000 | Acc: 0.6913
Epoch 7/20 | 185s | Loss: 0.5556 | MCC: 0.3245 | Acc: 0.7402
Epoch 8/20 | 186s | Loss: 0.5393 | MCC: 0.2869 | Acc: 0.7325
Epoch 9/20 | 186s | Loss: 0.5197 | MCC: 0.3165 | Acc: 0.7383
Epoch 10/20 | 185s | Loss: 0.5088 | MCC: 0.3100 | Acc: 0.7354
Epoch 11/20 | 186s | Loss: 0.4947 | MCC: 0.3160 | Acc: 0.7402
Epoch 12/20 | 185s | Loss: 0.4779 | MCC: 0.3385 | Acc: 0.7469
Epoch 13/20 | 186s | Loss: 0.4703 | MCC: 0.3367 | Acc: 0.7459
Epoch 14/20 | 186s | Loss: 0.4608 | MCC: 0.3522 | Acc: 0.7507
Epoch 15/20 | 186s | Loss: 0.4554 |

In [11]:
import gc; gc.collect(); torch.cuda.empty_cache()

set_seed(config.seed)
print("Training AdaLoRA (manual, rank=12→prune)")

base_model_ada = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).float()
model_ada = LoRAModel(base_model_ada, ["query_proj","key_proj","value_proj"],
    rank=config.adalora_init_rank, alpha=24, dropout=config.lora_dropout).to(device)

ada_lora_p = [p for n,p in model_ada.named_parameters() if 'lora_' in n and p.requires_grad]
ada_head_p = [p for n,p in model_ada.named_parameters() if ('classifier' in n or 'pooler' in n) and p.requires_grad]

optimizer_ada = AdamW([
    {'params': ada_lora_p, 'lr': 3e-4},
    {'params': ada_head_p, 'lr': 2e-3},
], weight_decay=config.weight_decay)

NUM_EPOCHS_ADA = 20
total_steps_ada = len(train_loader) * NUM_EPOCHS_ADA
scheduler_ada = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_ada, max_lr=[3e-4, 2e-3],
    total_steps=total_steps_ada, pct_start=config.warmup_ratio, anneal_strategy='cos')

ada_results = {'method':'adalora','train_losses':[],'val_losses':[],'val_mccs':[],'val_accs':[],'epoch_times':[],'best_mcc':-1.0,'best_epoch':0}
start_time = time.time()

for epoch in range(NUM_EPOCHS_ADA):
    t0 = time.time()
    model_ada.train()
    eloss, nb = 0.0, 0
    for batch in train_loader:
        batch = {k:v.to(device) for k,v in batch.items()}
        out = model_ada(input_ids=batch['input_ids'],attention_mask=batch['attention_mask'],labels=batch['labels'])
        if torch.isnan(out.loss): optimizer_ada.zero_grad(); continue
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model_ada.parameters() if p.requires_grad], 1.0)
        optimizer_ada.step(); scheduler_ada.step(); optimizer_ada.zero_grad()
        eloss += out.loss.item(); nb += 1
    val = evaluate(model_ada, val_loader, device)
    ada_results['train_losses'].append(eloss/max(nb,1))
    ada_results['val_losses'].append(val['loss'])
    ada_results['val_mccs'].append(val['mcc'])
    ada_results['val_accs'].append(val['accuracy'])
    ada_results['epoch_times'].append(time.time()-t0)
    if val['mcc']>ada_results['best_mcc']:
        ada_results['best_mcc']=val['mcc']; ada_results['best_epoch']=epoch+1
    print(f"Epoch {epoch+1}/{NUM_EPOCHS_ADA} | {time.time()-t0:.0f}s | Loss: {eloss/max(nb,1):.4f} | MCC: {val['mcc']:.4f} | Acc: {val['accuracy']:.4f}")

ada_results['total_time'] = time.time()-start_time
ada_results['effective_ranks'] = compute_effective_rank(model_ada, 'adalora')
ada_results['param_counts'] = count_parameters(model_ada, 'adalora')
print(f"\n✓ AdaLoRA Best MCC: {ada_results['best_mcc']:.4f} (epoch {ada_results['best_epoch']})")

Training AdaLoRA (manual, rank=12→prune)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

Injected LoRA into 36 layers
trainable params: 1,255,682 || all params: 185,087,234 || trainable%: 0.6784
Epoch 1/20 | 190s | Loss: 0.6195 | MCC: 0.1010 | Acc: 0.6951
Epoch 2/20 | 186s | Loss: 0.6302 | MCC: 0.0000 | Acc: 0.6913
Epoch 3/20 | 186s | Loss: 0.6164 | MCC: 0.0000 | Acc: 0.6913
Epoch 4/20 | 186s | Loss: 0.6127 | MCC: 0.0000 | Acc: 0.6913
Epoch 5/20 | 186s | Loss: 0.6131 | MCC: 0.0000 | Acc: 0.6913
Epoch 6/20 | 186s | Loss: 0.5919 | MCC: 0.0000 | Acc: 0.6913
Epoch 7/20 | 186s | Loss: 0.5699 | MCC: 0.3122 | Acc: 0.7373
Epoch 8/20 | 186s | Loss: 0.5439 | MCC: 0.3475 | Acc: 0.7402
Epoch 9/20 | 186s | Loss: 0.5137 | MCC: 0.3561 | Acc: 0.7488
Epoch 10/20 | 186s | Loss: 0.4824 | MCC: 0.3350 | Acc: 0.7421
Epoch 11/20 | 186s | Loss: 0.4685 | MCC: 0.3414 | Acc: 0.7478
Epoch 12/20 | 186s | Loss: 0.4443 | MCC: 0.3573 | Acc: 0.7526
Epoch 13/20 | 186s | Loss: 0.4277 | MCC: 0.3724 | Acc: 0.7565
Epoch 14/20 | 186s | Loss: 0.4116 | MCC: 0.3891 | Acc: 0.7622
Epoch 15/20 | 186s | Loss: 0.4063 |

In [41]:
import gc
for v in ['model_ada','base_model_ada','optimizer_ada','scheduler_ada']:
    if v in globals(): exec(f'del {v}')
gc.collect(); torch.cuda.empty_cache()

set_seed(config.seed)
print("Training Real AdaLoRA")

base_model_ada = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS).float()

model_ada = AdaLoRAModel(base_model_ada, config.target_modules,
    init_rank=config.adalora_init_rank, alpha=24, dropout=config.lora_dropout).to(device)

for n, p in model_ada.base_model.named_parameters():
    if 'classifier' in n or 'pooler' in n:
        p.requires_grad = True

ada_params = model_ada.get_adalora_parameters()     
lam_params = model_ada.get_lambda_parameters()        
head_params = [p for n, p in model_ada.base_model.named_parameters()
               if p.requires_grad and ('classifier' in n or 'pooler' in n)]

trainable = sum(p.numel() for p in ada_params) + sum(p.numel() for p in lam_params)
total = sum(p.numel() for p in model_ada.base_model.parameters())
print(f"Trainable: {trainable:,} ({100*trainable/total:.4f}%)")

optimizer_ada = AdamW([
    {'params': ada_params, 'lr': 3e-4},
    {'params': lam_params, 'lr': 5e-4},   
    {'params': head_params, 'lr': 2e-3},
], weight_decay=config.weight_decay)

NUM_EPOCHS = 20
total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * 0.1)
scheduler_ada = torch.optim.lr_scheduler.LambdaLR(
    optimizer_ada, lr_lambda=lambda step: min(1.0, step / max(warmup_steps, 1)))


INIT_TOTAL = config.adalora_init_rank * 36   
TARGET_TOTAL = 6 * 36                         
PRUNE_START = 8                                
PRUNE_END = 16                                 

ada_results = {'method':'adalora','train_losses':[],'val_mccs':[],'val_accs':[],
    'epoch_times':[],'best_mcc':-1.0,'best_epoch':0,'effective_rank_history':[]}
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    model_ada.train()
    epoch_loss, num_batches = 0.0, 0

    for batch in train_loader:
        optimizer_ada.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model_ada(input_ids=batch['input_ids'],
                            attention_mask=batch['attention_mask'], labels=batch['labels'])
        loss = outputs.loss
        if torch.isnan(loss): continue
        loss.backward()

        model_ada.update_all_importance()

        torch.nn.utils.clip_grad_norm_(ada_params + lam_params, config.max_grad_norm)
        optimizer_ada.step()
        scheduler_ada.step()
        epoch_loss += loss.item()
        num_batches += 1

    # Pruning schedule: linearly reduce budget from INIT_TOTAL to TARGET_TOTAL
    if PRUNE_START <= epoch < PRUNE_END:
        progress = (epoch - PRUNE_START) / (PRUNE_END - PRUNE_START)
        budget = int(INIT_TOTAL - progress * (INIT_TOTAL - TARGET_TOTAL))
        model_ada.prune_to_budget(budget)

    val = evaluate(model_ada, val_loader, device)
    avg_rank = model_ada.get_avg_effective_rank()

    ada_results['train_losses'].append(epoch_loss/num_batches)
    ada_results['val_mccs'].append(val['mcc'])
    ada_results['val_accs'].append(val['accuracy'])
    ada_results['epoch_times'].append(time.time()-epoch_start)
    ada_results['effective_rank_history'].append(avg_rank)

    if val['mcc'] > ada_results['best_mcc']:
        ada_results['best_mcc'] = val['mcc']
        ada_results['best_epoch'] = epoch + 1

    phase = "WARMUP" if epoch < PRUNE_START else ("PRUNE" if epoch < PRUNE_END else "FINETUNE")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} [{phase}] | {time.time()-epoch_start:.0f}s | "
          f"Loss: {epoch_loss/num_batches:.4f} | MCC: {val['mcc']:.4f} | "
          f"EffRank: {avg_rank:.1f}/{config.adalora_init_rank}")

ada_results['total_time'] = time.time() - start_time
ada_results['effective_ranks'] = model_ada.get_effective_ranks()
print(f"\nAdaLoRA Best MCC: {ada_results['best_mcc']:.4f} (epoch {ada_results['best_epoch']})")
print(f"Final avg effective rank: {model_ada.get_avg_effective_rank():.1f}/{config.adalora_init_rank}")

Training Real AdaLoRA


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

Injected AdaLoRA into 36 layers (init_rank=12)
Trainable: 432 (0.0002%)
Epoch 1/20 [WARMUP] | 187s | Loss: 0.6259 | MCC: 0.0000 | EffRank: 12.0/12
Epoch 2/20 [WARMUP] | 186s | Loss: 0.6145 | MCC: 0.0000 | EffRank: 12.0/12
Epoch 3/20 [WARMUP] | 186s | Loss: 0.6200 | MCC: 0.0000 | EffRank: 12.0/12
Epoch 4/20 [WARMUP] | 186s | Loss: 0.6164 | MCC: 0.0000 | EffRank: 12.0/12
Epoch 5/20 [WARMUP] | 187s | Loss: 0.6158 | MCC: 0.0656 | EffRank: 12.0/12
Epoch 6/20 [WARMUP] | 187s | Loss: 0.6147 | MCC: 0.0000 | EffRank: 12.0/12
Epoch 7/20 [WARMUP] | 186s | Loss: 0.6119 | MCC: 0.0000 | EffRank: 12.0/12
Epoch 8/20 [WARMUP] | 186s | Loss: 0.6086 | MCC: 0.0000 | EffRank: 12.0/12
Epoch 9/20 [PRUNE] | 187s | Loss: 0.6095 | MCC: 0.0000 | EffRank: 12.0/12
Epoch 10/20 [PRUNE] | 187s | Loss: 0.6089 | MCC: 0.0000 | EffRank: 11.2/12
Epoch 11/20 [PRUNE] | 186s | Loss: 0.6088 | MCC: 0.0000 | EffRank: 10.5/12
Epoch 12/20 [PRUNE] | 186s | Loss: 0.6090 | MCC: 0.0000 | EffRank: 9.8/12
Epoch 13/20 [PRUNE] | 186s | L

KeyboardInterrupt: 

In [40]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class AdaLoRALinear(nn.Module):
    def __init__(self, in_features, out_features, init_rank=12, alpha=16.0, dropout=0.1):
        super().__init__()
        self.init_rank = init_rank
        self.scaling   = alpha / init_rank

        self.P      = nn.Parameter(torch.empty(out_features, init_rank))
        self.Lambda = nn.Parameter(torch.empty(init_rank))          
        self.Q      = nn.Parameter(torch.empty(init_rank, in_features))

        self.lora_dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()

        self.register_buffer('rank_mask',  torch.ones(init_rank))
        self.register_buffer('importance', torch.zeros(init_rank))

        nn.init.orthogonal_(self.P)                              
        nn.init.orthogonal_(self.Q)                                 

        nn.init.uniform_(self.Lambda, a=0.01, b=0.1)               

    
    def forward(self, x):
        masked_lambda = self.Lambda * self.rank_mask                
        out = F.linear(self.lora_dropout(x), self.Q)               
        out = out * masked_lambda                                  
        out = F.linear(out, self.P)                                 
        return out * self.scaling

    def update_importance(self, beta=0.85):
        if self.Lambda.grad is not None:
            new_imp = self.Lambda.grad.abs()                        # ← FIX 3
            self.importance = beta * self.importance + (1 - beta) * new_imp

    def get_effective_rank(self):
        return int(self.rank_mask.sum().item())



class SoRAWrappedLinear(nn.Module):
    def __init__(self, original_linear, adapter_layer):
        super().__init__()
        self.original = original_linear
        self.sora     = adapter_layer

    def forward(self, x):
        return self.original(x) + self.sora(x)


def _set_nested_attr(obj, attr_path, value):
    parts = attr_path.split('.')
    for part in parts[:-1]:
        obj = getattr(obj, part)
    setattr(obj, parts[-1], value)

class AdaLoRAModel(nn.Module):
    def __init__(self, base_model, target_modules,
                 init_rank=12, alpha=16.0, dropout=0.1):
        super().__init__()
        self.base_model     = base_model
        
        self.adalora_layers = nn.ModuleDict()                      

        for param in self.base_model.parameters():
            param.requires_grad = False

        count = 0
        for name, module in list(self.base_model.named_modules()):
            if isinstance(module, nn.Linear) and any(t in name for t in target_modules):
                ada = AdaLoRALinear(module.in_features, module.out_features,
                                    init_rank, alpha, dropout)
                key = name.replace('.', '_')
                self.adalora_layers[key] = ada                      # into ModuleDict
                wrapped = SoRAWrappedLinear(module, ada)
                _set_nested_attr(self.base_model, name, wrapped)
                count += 1

        print(f"Injected AdaLoRA into {count} layers (init_rank={init_rank})")

    def forward(self, **kwargs):
        return self.base_model(**kwargs)

    def get_adalora_parameters(self):
        """P and Q only — not Lambda."""
        return [p for n, p in self.named_parameters()
                if 'adalora_layers' in n and 'Lambda' not in n]

    def get_lambda_parameters(self):
        return [layer.Lambda for layer in self.adalora_layers.values()]

    def update_all_importance(self):
        for layer in self.adalora_layers.values():
            layer.update_importance()

    def prune_to_budget(self, target_total_rank):
        all_scores = []
        for key, layer in self.adalora_layers.items():
            for i in range(layer.init_rank):
                if layer.rank_mask[i] > 0:
                    all_scores.append((layer.importance[i].item(), key, i))

        all_scores.sort(key=lambda x: x[0])                       
        current_total = sum(l.rank_mask.sum().item() for l in self.adalora_layers.values())
        n_to_prune = int(current_total - target_total_rank)

        for idx in range(max(0, min(n_to_prune, len(all_scores)))):
            _, key, dim_idx = all_scores[idx]
            self.adalora_layers[key].rank_mask[dim_idx]    = 0.0
            self.adalora_layers[key].Lambda.data[dim_idx]  = 0.0

    def get_effective_ranks(self):
        return {k: {'effective_rank': v.get_effective_rank(),
                    'max_rank':       v.init_rank,
                    'importance':     v.importance.tolist(),
                    'mask':           v.rank_mask.tolist()}
                for k, v in self.adalora_layers.items()}

    def get_avg_effective_rank(self):
        ranks = [v.get_effective_rank() for v in self.adalora_layers.values()]
        return sum(ranks) / len(ranks) if ranks else 0

    # ── quick sanity check ────────────────────────────────────────────────
    def verify_trainable(self):
        ada_p, lam_p, other_p = [], [], []
        for n, p in self.named_parameters():
            if not p.requires_grad:
                continue
            if 'Lambda' in n:
                lam_p.append((n, p.numel()))
            elif 'adalora_layers' in n:
                ada_p.append((n, p.numel()))
            else:
                other_p.append((n, p.numel()))

        print(f"\n{'─'*55}")
        print(f"  P/Q params  : {sum(x[1] for x in ada_p):>10,}  ({len(ada_p)} tensors)")
        print(f"  Lambda      : {sum(x[1] for x in lam_p):>10,}  ({len(lam_p)} tensors)")
        print(f"  Head/Pooler : {sum(x[1] for x in other_p):>10,}  ({len(other_p)} tensors)")
        total_trainable = sum(x[1] for x in ada_p + lam_p + other_p)
        total_all = sum(p.numel() for p in self.parameters())
        print(f"  TOTAL       : {total_trainable:>10,}  ({100*total_trainable/total_all:.4f}%)")
        print(f"{'─'*55}\n")

        assert sum(x[1] for x in ada_p) > 0,  "❌ P/Q not in optimizer — ModuleDict fix not working"
        assert sum(x[1] for x in lam_p) > 0,  "❌ Lambda missing"
        assert sum(x[1] for x in other_p) > 0, "❌ Classifier head not trainable"
        print("  ✅ All parameter groups confirmed trainable\n")

In [12]:
from sklearn.metrics import matthews_corrcoef, accuracy_score
import torch
import numpy as np


def evaluate(model, dataloader, device, is_sora: bool = False):
    with keys: loss, mcc, accuracy
    """
    model.eval()
    all_preds  = []
    all_labels = []
    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(
                input_ids      = batch['input_ids'],
                attention_mask = batch['attention_mask'],
                labels         = batch['labels'],
            )

            loss   = outputs.loss
            logits = outputs.logits   

            if not torch.isnan(loss):
                total_loss += loss.item()
                num_batches += 1

            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(batch['labels'].cpu().numpy().tolist())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    avg_loss = total_loss / max(num_batches, 1)
    mcc      = matthews_corrcoef(all_labels, all_preds)
    acc      = accuracy_score(all_labels, all_preds)

    return {
        'loss':     avg_loss,
        'mcc':      mcc,
        'accuracy': acc,
    }

In [13]:
import gc
for v in ['model_sora','base_model_sora','optimizer_ab','proximal_optimizer','scheduler_sora']:
    if v in globals(): exec(f'del {v}')
gc.collect(); torch.cuda.empty_cache()

set_seed(config.seed)
print("Training SoRA")

base_model_sora = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).float()
# NO gradient_checkpointing_enable — it breaks adapter gradients

model_sora = SoRAModel(base_model_sora, config.target_modules, rank=config.sora_rank,
    alpha=config.lora_alpha, dropout=config.lora_dropout).to(device)

for n, p in model_sora.base_model.named_parameters():
    if 'classifier' in n or 'pooler' in n:
        p.requires_grad = True

param_info = model_sora.count_trainable_parameters()
print(f"Trainable: {param_info['trainable_params']:,} ({param_info['trainable_percent']:.4f}%)")

sora_params = model_sora.get_sora_parameters()
gate_params = list(model_sora.get_gate_parameters())
head_params = [p for n, p in model_sora.base_model.named_parameters()
               if p.requires_grad and ('classifier' in n or 'pooler' in n)]

optimizer_ab = AdamW([
    {'params': sora_params, 'lr': 3e-4},
    {'params': head_params, 'lr': 2e-3},
], weight_decay=config.weight_decay)

proximal_optimizer = ProximalSGD(gate_params, lr=1e-2, lam=0.0)

NUM_EPOCHS = 20
total_steps = len(train_loader) * NUM_EPOCHS

# Constant LR after warmup — no decay to zero
warmup_steps = int(total_steps * 0.1)
scheduler_sora = torch.optim.lr_scheduler.LambdaLR(
    optimizer_ab, lr_lambda=lambda step: min(1.0, step / max(warmup_steps, 1)))

sora_results = {'method':'sora','train_losses':[],'val_losses':[],'val_mccs':[],'val_accs':[],
    'epoch_times':[],'best_mcc':-1.0,'best_epoch':0,'effective_rank_history':[],'gate_sparsity_history':[]}
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    model_sora.train()
    epoch_loss, epoch_l1, num_batches = 0.0, 0.0, 0

    # Phase 1: learn (epochs 1-10), Phase 2: prune (epochs 11-20)
    if epoch < 10:
        proximal_optimizer.lam = 0.0
        phase = "LEARN"
    else:
        proximal_optimizer.lam = 0.05 * min(1.0, (epoch - 9) / 5)
        phase = "PRUNE"

    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model_sora(input_ids=batch['input_ids'],
                             attention_mask=batch['attention_mask'], labels=batch['labels'])
        task_loss = outputs.loss
        if torch.isnan(task_loss):
            optimizer_ab.zero_grad(); continue

        task_loss.backward()
        torch.nn.utils.clip_grad_norm_(sora_params, config.max_grad_norm)

        optimizer_ab.step()
        scheduler_sora.step()

        # Proximal BEFORE zero_grad (gate grads still alive)
        proximal_optimizer.step()

        optimizer_ab.zero_grad()
        proximal_optimizer.zero_grad()

        epoch_loss += task_loss.item()
        with torch.no_grad():
            epoch_l1 += compute_sora_l1_loss(model_sora).item() * proximal_optimizer.lam
        num_batches += 1

    val = evaluate(model_sora, val_loader, device, is_sora=True)
    avg_rank = model_sora.get_avg_effective_rank()
    all_ranks = model_sora.get_effective_ranks()
    avg_sparsity = np.mean([s['gate_sparsity'] for s in all_ranks.values()])

    sora_results['train_losses'].append(epoch_loss/num_batches)
    sora_results['val_losses'].append(val['loss'])
    sora_results['val_mccs'].append(val['mcc'])
    sora_results['val_accs'].append(val['accuracy'])
    sora_results['epoch_times'].append(time.time()-epoch_start)
    sora_results['effective_rank_history'].append(avg_rank)
    sora_results['gate_sparsity_history'].append(avg_sparsity)

    if val['mcc'] > sora_results['best_mcc']:
        sora_results['best_mcc'] = val['mcc']
        sora_results['best_epoch'] = epoch + 1

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} [{phase}] | {time.time()-epoch_start:.0f}s | "
          f"Loss: {epoch_loss/num_batches:.4f} | L1: {epoch_l1/num_batches:.4f} | "
          f"λ: {proximal_optimizer.lam:.4f} | MCC: {val['mcc']:.4f} | "
          f"EffRank: {avg_rank:.1f}/{config.sora_rank} | Sparsity: {avg_sparsity:.2%}")

sora_results['total_time'] = time.time() - start_time
sora_results['effective_ranks'] = model_sora.get_effective_ranks()
sora_results['param_counts'] = model_sora.count_trainable_parameters()
print(f"\nSoRA Best MCC: {sora_results['best_mcc']:.4f} (epoch {sora_results['best_epoch']})")

Training SoRA


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

Injected SoRA adapters into 36 layers
Trainable: 442,656 (0.2400%)
Epoch 1/20 [LEARN] | 187s | Loss: 0.6244 | L1: 0.0000 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 2/20 [LEARN] | 186s | Loss: 0.6180 | L1: 0.0000 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 3/20 [LEARN] | 186s | Loss: 0.6151 | L1: 0.0000 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 4/20 [LEARN] | 186s | Loss: 0.6127 | L1: 0.0000 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 5/20 [LEARN] | 186s | Loss: 0.6084 | L1: 0.0000 | λ: 0.0000 | MCC: -0.0232 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 6/20 [LEARN] | 187s | Loss: 0.6025 | L1: 0.0000 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 7/20 [LEARN] | 186s | Loss: 0.5919 | L1: 0.0000 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 8/20 [LEARN] | 186s | Loss: 0.5757 | L1: 0.0000 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 9/20 [

In [14]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

all_results = {'lora': lora_results, 'adalora': ada_results, 'sora': sora_results}

print("=" * 80)
print("COMPARISON RESULTS")
print("=" * 80)
print(f"\n{'Method':<12} {'Best MCC':>10} {'Best Epoch':>12} {'Trainable':>12} {'Train%':>10} {'Total Time':>12}")
print("-" * 80)
for method, res in all_results.items():
    params = res['param_counts']
    print(f"{method:<12} {res['best_mcc']:>10.4f} {res['best_epoch']:>12d} {params['trainable_params']:>12,} {params['trainable_percent']:>9.4f}% {res['total_time']:>11.1f}s")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LoRA vs AdaLoRA vs SoRA on CoLA (DeBERTaV3-base)', fontsize=14, fontweight='bold')
colors = {'lora': '#2196F3', 'adalora': '#FF9800', 'sora': '#4CAF50'}

ax = axes[0, 0]
for method, res in all_results.items():
    epochs = range(1, len(res['val_mccs']) + 1)
    ax.plot(epochs, res['val_mccs'], '-o', label=method.upper(), color=colors[method], markersize=4)
ax.set_xlabel('Epoch'); ax.set_ylabel('MCC'); ax.set_title('(a) Performance: MCC on CoLA Dev')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
methods = list(all_results.keys())
param_counts = [all_results[m]['param_counts']['trainable_params'] for m in methods]
param_pcts = [all_results[m]['param_counts']['trainable_percent'] for m in methods]
bars = ax.bar([m.upper() for m in methods], param_counts, color=[colors[m] for m in methods])
for bar, pct in zip(bars, param_pcts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{pct:.3f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Trainable Parameters'); ax.set_title('(b) Number of Trainable Parameters')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1, 0]
for method, res in all_results.items():
    epochs = range(1, len(res['train_losses']) + 1)
    ax.plot(epochs, res['train_losses'], '-o', label=method.upper(), color=colors[method], markersize=4)
ax.set_xlabel('Epoch'); ax.set_ylabel('Training Loss'); ax.set_title('(c) Training Loss Convergence')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
for method, res in all_results.items():
    epochs = range(1, len(res['epoch_times']) + 1)
    ax.plot(epochs, res['epoch_times'], '-o', label=method.upper(), color=colors[method], markersize=4)
ax.set_xlabel('Epoch'); ax.set_ylabel('Time (seconds)'); ax.set_title('(d) Training Efficiency')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'comparison_results.png'), dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('SoRA Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
epochs = range(1, len(sora_results['effective_rank_history']) + 1)
ax.plot(epochs, sora_results['effective_rank_history'], '-o', color='#4CAF50', markersize=5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Avg Effective Rank'); ax.set_title('Effective Rank Evolution'); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(epochs, sora_results['gate_sparsity_history'], '-o', color='#F44336', markersize=5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Gate Sparsity'); ax.set_title('Sparsity Over Training'); ax.grid(True, alpha=0.3)

ax = axes[2]
all_gates = [v['gate_mean'] for v in sora_results['effective_ranks'].values()]
ax.bar(range(len(all_gates)), all_gates, color='#9C27B0', alpha=0.7)
ax.set_xlabel('Layer Index'); ax.set_ylabel('Mean |gate| value'); ax.set_title('Final Gate Magnitudes by Layer'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'sora_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save results JSON
import json
def convert(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    elif isinstance(obj, (np.floating,)): return float(obj)
    elif isinstance(obj, np.ndarray): return obj.tolist()
    return obj

with open(os.path.join(OUTPUT_DIR, 'comparison_results.json'), 'w') as f:
    json.dump({m: {k: convert(v) if not isinstance(v, (dict, list)) else v for k, v in r.items()} for m, r in all_results.items()}, f, indent=2, default=str)
print("Results saved!")

COMPARISON RESULTS

Method         Best MCC   Best Epoch    Trainable     Train%   Total Time
--------------------------------------------------------------------------------
lora             0.3574           18    1,034,498    0.5596%      3706.9s
adalora          0.3919           16    1,255,682    0.6784%      3724.5s
sora             0.3711           12      442,656    0.2400%      3725.9s
Results saved!


In [15]:
print("=" * 70)
print("Part 2: SGD Subgradient vs Proximal Gradient Descent")
print("=" * 70)

def numpy_sora_forward(x, A, B, g, scaling=1.0):
   
    down = x @ A.T         
    gated = down * g        
    up = gated @ B.T         
    return up * scaling

def numpy_sora_backward(x, A, B, g, grad_output, scaling=1.0):
   
    grad_output_scaled = grad_output * scaling
    
    down = x @ A.T
    gated = down * g
    grad_B = grad_output_scaled.T @ gated 
    
    grad_gated = grad_output_scaled @ B  
    grad_g = np.sum(grad_gated * down, axis=0)  
    
    grad_down = grad_gated * g  
    grad_A = grad_down.T @ x  
    
    return grad_A, grad_B, grad_g

def numpy_sgd_subgradient_update(g, grad_g_task, lr, lam):
    
    subgrad = np.sign(g)  # sign(0) = 0 in numpy
    total_grad = grad_g_task + lam * subgrad
    g_new = g - lr * total_grad
    return g_new

def numpy_proximal_update(g, grad_g_task, lr, lam):
    
    g_temp = g - lr * grad_g_task
    xi = lr * lam
    g_new = np.sign(g_temp) * np.maximum(np.abs(g_temp) - xi, 0.0)
    return g_new

# Test on a simple problem
np.random.seed(42)
d_in, d_out, r, batch_size = 16, 16, 4, 8

A = np.random.randn(r, d_in) * 0.1
B = np.zeros((d_out, r))
g_prox = np.ones(r)
g_sgd = np.ones(r)

x = np.random.randn(batch_size, d_in)
target = np.random.randn(batch_size, d_out) * 0.1

lr = 0.01
lam = 0.1
n_steps = 200

prox_history = {'g': [], 'loss': [], 'sparsity': []}
sgd_history = {'g': [], 'loss': [], 'sparsity': []}

for step in range(n_steps):
    out_prox = numpy_sora_forward(x, A, B, g_prox)
    out_sgd = numpy_sora_forward(x, A, B, g_sgd)
    
    grad_out_prox = 2 * (out_prox - target) / batch_size
    grad_out_sgd = 2 * (out_sgd - target) / batch_size
    
    _, _, grad_g_prox = numpy_sora_backward(x, A, B, g_prox, grad_out_prox)
    _, _, grad_g_sgd = numpy_sora_backward(x, A, B, g_sgd, grad_out_sgd)
    
    g_prox = numpy_proximal_update(g_prox, grad_g_prox, lr, lam)
    g_sgd = numpy_sgd_subgradient_update(g_sgd, grad_g_sgd, lr, lam)
    
    loss_prox = np.mean((out_prox - target) ** 2) + lam * np.sum(np.abs(g_prox))
    loss_sgd = np.mean((out_sgd - target) ** 2) + lam * np.sum(np.abs(g_sgd))
    
    prox_history['g'].append(g_prox.copy())
    prox_history['loss'].append(loss_prox)
    prox_history['sparsity'].append(np.sum(np.abs(g_prox) < 1e-5) / r)
    
    sgd_history['g'].append(g_sgd.copy())
    sgd_history['loss'].append(loss_sgd)
    sgd_history['sparsity'].append(np.sum(np.abs(g_sgd) < 1e-5) / r)

print(f"Final g (Proximal): {g_prox}")
print(f"Final g (SGD Sub):  {g_sgd}")
print(f"Proximal sparsity: {prox_history['sparsity'][-1]:.2%}")
print(f"SGD sparsity:      {sgd_history['sparsity'][-1]:.2%}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('NumPy: Proximal vs SGD Subgradient', fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(prox_history['loss'], label='Proximal', color='#4CAF50')
ax.plot(sgd_history['loss'], label='SGD Subgrad', color='#F44336')
ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.set_title('Loss Convergence')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(prox_history['sparsity'], label='Proximal', color='#4CAF50')
ax.plot(sgd_history['sparsity'], label='SGD Subgrad', color='#F44336')
ax.set_xlabel('Step'); ax.set_ylabel('Sparsity'); ax.set_title('Gate Sparsity Over Steps')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
for i in range(r):
    prox_g_vals = [h[i] for h in prox_history['g']]
    sgd_g_vals = [h[i] for h in sgd_history['g']]
    ax.plot(prox_g_vals, '--', alpha=0.7, color=f'C{i}')
    ax.plot(sgd_g_vals, '-', alpha=0.7, color=f'C{i}')
ax.set_xlabel('Step'); ax.set_ylabel('Gate Value'); ax.set_title('Gate Evolution (-- prox, — sgd)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'numpy_prox_vs_sgd.png'), dpi=150, bbox_inches='tight')
plt.show()

Part 2: SGD Subgradient vs Proximal Gradient Descent
Final g (Proximal): [0.8 0.8 0.8 0.8]
Final g (SGD Sub):  [0.8 0.8 0.8 0.8]
Proximal sparsity: 0.00%
SGD sparsity:      0.00%


In [18]:
import numpy as np
import matplotlib.pyplot as plt
import os

PLOTS_DIR = "."

print("=" * 70)
print("Part 2: SGD Subgradient vs Proximal Gradient Descent")
print("=" * 70)


def numpy_sora_forward(x, A, B, g, scaling=1.0):
    down  = x @ A.T      
    gated = down * g
    up    = gated @ B.T    
    return up * scaling

def numpy_sora_backward(x, A, B, g, grad_output, scaling=1.0):
    g_out  = grad_output * scaling
    down   = x @ A.T
    gated  = down * g
    grad_B     = g_out.T @ gated            
    grad_gated = g_out @ B                  
    grad_g     = np.sum(grad_gated * down, axis=0)  
    grad_A     = (grad_gated * g).T @ x    
    return grad_A, grad_B, grad_g


def sgd_subgradient_update(g, grad_g, lr, lam):
    """
    SGD with L1 subgradient:
        g ← g - lr * (∇L_task + λ·sign(g))
    sign(0) = 0 in NumPy — canonical subgradient choice.
    Does NOT guarantee exact zeros; overshoots and oscillates near zero.
    """
    return g - lr * (grad_g + lam * np.sign(g))

def proximal_update(g, grad_g, lr, lam):
    """
    Proximal gradient step:
        g_half = g - lr * ∇L_task          (smooth gradient step)
        g_new  = sign(g_half) * max(|g_half| - lr*λ, 0)  (soft-threshold)
    Guarantees exact zeros: any |g_half_i| ≤ lr*λ maps to exactly 0.
    """
    g_half = g - lr * grad_g
    return np.sign(g_half) * np.maximum(np.abs(g_half) - lr * lam, 0.0)


np.random.seed(42)
d_in, d_out, r, batch_size = 16, 16, 8, 8

A_true = np.random.randn(4, d_in) * 0.3
B_true = np.random.randn(d_out, 4) * 0.3
x      = np.random.randn(batch_size, d_in)
target = x @ A_true.T @ B_true.T    

A_init = np.random.randn(r, d_in) * 0.1
B_init = np.random.randn(d_out, r) * 0.1  

A_prox, B_prox, g_prox = A_init.copy(), B_init.copy(), np.ones(r)
A_sgd,  B_sgd,  g_sgd  = A_init.copy(), B_init.copy(), np.ones(r)

lr    = 0.01   
lam   = 0.3    
       

n_steps = 500

prox_history = {'g': [], 'loss': [], 'sparsity': []}
sgd_history  = {'g': [], 'loss': [], 'sparsity': []}

# ── Training loop ─────────────────────────────────────────────────────────

for step in range(n_steps):

    # ── Proximal ──────────────────────────────────────────────────────────
    out_prox      = numpy_sora_forward(x, A_prox, B_prox, g_prox)
    grad_out_prox = 2 * (out_prox - target) / batch_size
    grad_A_prox, grad_B_prox, grad_g_prox = numpy_sora_backward(
        x, A_prox, B_prox, g_prox, grad_out_prox)

    A_prox -= lr * grad_A_prox
    B_prox -= lr * grad_B_prox
    g_prox  = proximal_update(g_prox, grad_g_prox, lr, lam)

    # ── SGD subgradient ───────────────────────────────────────────────────
    out_sgd      = numpy_sora_forward(x, A_sgd, B_sgd, g_sgd)
    grad_out_sgd = 2 * (out_sgd - target) / batch_size
    grad_A_sgd, grad_B_sgd, grad_g_sgd = numpy_sora_backward(
        x, A_sgd, B_sgd, g_sgd, grad_out_sgd)

    A_sgd -= lr * grad_A_sgd
    B_sgd -= lr * grad_B_sgd
    g_sgd  = sgd_subgradient_update(g_sgd, grad_g_sgd, lr, lam)

    # ── Track ─────────────────────────────────────────────────────────────
    loss_prox = np.mean((out_prox - target)**2) + lam * np.sum(np.abs(g_prox))
    loss_sgd  = np.mean((out_sgd  - target)**2) + lam * np.sum(np.abs(g_sgd))

    prox_history['g'].append(g_prox.copy())
    prox_history['loss'].append(loss_prox)
    prox_history['sparsity'].append(np.sum(np.abs(g_prox) < 1e-5) / r)

    sgd_history['g'].append(g_sgd.copy())
    sgd_history['loss'].append(loss_sgd)
    sgd_history['sparsity'].append(np.sum(np.abs(g_sgd) < 1e-5) / r)

# ── Results ───────────────────────────────────────────────────────────────

print(f"\nFinal g (Proximal): {np.round(g_prox, 6)}")
print(f"Final g (SGD Sub):  {np.round(g_sgd,  6)}")
print(f"\nProximal sparsity:  {prox_history['sparsity'][-1]:.2%}")
print(f"SGD sparsity:       {sgd_history['sparsity'][-1]:.2%}")
print(f"\nFinal loss (Proximal): {prox_history['loss'][-1]:.6f}")
print(f"Final loss (SGD Sub):  {sgd_history['loss'][-1]:.6f}")

# ── Plot ──────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('NumPy: Proximal vs SGD Subgradient  '
             f'(rank-4 target, r=8, λ={lam})',
             fontsize=13, fontweight='bold')

# Loss
ax = axes[0]
ax.plot(prox_history['loss'], label='Proximal',    color='#4CAF50', linewidth=2)
ax.plot(sgd_history['loss'],  label='SGD Subgrad', color='#F44336', linewidth=2,
        linestyle='--')
ax.set_xlabel('Step'); ax.set_ylabel('Loss (MSE + λ||g||₁)')
ax.set_title('Loss Convergence')
ax.legend(); ax.grid(True, alpha=0.3)

# Sparsity
ax = axes[1]
ax.plot(prox_history['sparsity'], label='Proximal',    color='#4CAF50', linewidth=2)
ax.plot(sgd_history['sparsity'],  label='SGD Subgrad', color='#F44336', linewidth=2,
        linestyle='--')
ax.set_xlabel('Step'); ax.set_ylabel('Fraction of |gᵢ| < 1e-5')
ax.set_title('Gate Sparsity Over Steps')
ax.set_ylim(-0.05, 1.05)
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
colors = plt.cm.tab10(np.linspace(0, 1, r))
for i in range(r):
    pv = [h[i] for h in prox_history['g']]
    sv = [h[i] for h in sgd_history['g']]
    ax.plot(pv, '--', alpha=0.85, color=colors[i], linewidth=1.5)
    ax.plot(sv, '-',  alpha=0.85, color=colors[i], linewidth=1.5)

ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax.set_xlabel('Step'); ax.set_ylabel('Gate value')
ax.set_title('Gate Evolution  (-- proximal,  — SGD)')

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0], [0], color='gray', linestyle='--', linewidth=2, label='Proximal'),
    Line2D([0], [0], color='gray', linestyle='-',  linewidth=2, label='SGD subgrad'),
], loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(PLOTS_DIR, 'numpy_prox_vs_sgd.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\nPlot saved to {out_path}")

Part 2: SGD Subgradient vs Proximal Gradient Descent

Final g (Proximal): [0.496566 0.       0.119561 0.       0.       0.428551 0.569161 0.      ]
Final g (SGD Sub):  [ 4.96567e-01 -1.12400e-03  1.19429e-01  1.53300e-03 -2.56200e-03
  4.28382e-01  5.68897e-01 -6.50000e-05]

Proximal sparsity:  50.00%
SGD sparsity:       0.00%

Final loss (Proximal): 0.492252
Final loss (SGD Sub):  0.493645

Plot saved to ./numpy_prox_vs_sgd.png


In [17]:
import gc; gc.collect(); torch.cuda.empty_cache()
torch.cuda.empty_cache()

set_seed(config.seed)
print("=" * 70)
print("Training SoRA with SGD Subgradient (instead of Proximal)")
print("=" * 70)

base_model_sgd = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).float()
model_sgd = SoRAModel(base_model_sgd, config.target_modules, rank=config.sora_rank, alpha=config.lora_alpha, dropout=config.lora_dropout).to(device)

for n, p in model_sgd.base_model.named_parameters():
    if 'classifier' in n or 'pooler' in n:
        p.requires_grad = True

sora_params_sgd = model_sgd.get_sora_parameters()
gate_params_sgd = model_sgd.get_gate_parameters()
classifier_params_sgd = [p for n, p in model_sgd.base_model.named_parameters() if p.requires_grad and 'classifier' in n]

optimizer_ab_sgd = AdamW(sora_params_sgd + classifier_params_sgd, lr=config.learning_rate, weight_decay=config.weight_decay)
sgd_subgrad_optimizer = SGDSubgradientOptimizer(gate_params_sgd, lr=config.sora_proximal_lr, lam=config.sora_lambda)
total_steps = len(train_loader) * config.num_epochs
scheduler_sgd = torch.optim.lr_scheduler.OneCycleLR(optimizer_ab_sgd, max_lr=config.learning_rate, total_steps=total_steps, pct_start=config.warmup_ratio, anneal_strategy='cos')

sgd_results = {'method': 'sora_sgd', 'train_losses': [], 'val_losses': [], 'val_mccs': [], 'val_accs': [], 'epoch_times': [], 'best_mcc': -1.0, 'best_epoch': 0, 'effective_rank_history': [], 'gate_sparsity_history': []}
start_time = time.time()

for epoch in range(config.num_epochs):
    epoch_start = time.time()
    model_sgd.train()
    epoch_loss, num_batches = 0.0, 0
    
    lambda_scale = min(1.0, (epoch + 1) / (config.num_epochs * 0.5))
    current_lambda = config.sora_lambda * lambda_scale
    sgd_subgrad_optimizer.lam = current_lambda
    
    for batch_idx, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model_sgd(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=batch['labels'])
        
        task_loss = outputs.loss
        l1_loss = compute_sora_l1_loss(model_sgd) * current_lambda
        total_loss = task_loss + l1_loss
        total_loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model_sgd.get_sora_parameters(), config.max_grad_norm)
        optimizer_ab_sgd.step()
        scheduler_sgd.step()
        optimizer_ab_sgd.zero_grad()
        
        sgd_subgrad_optimizer.step()
        sgd_subgrad_optimizer.zero_grad()
        
        epoch_loss += task_loss.item()
        num_batches += 1
    
    epoch_time = time.time() - epoch_start
    val_metrics = evaluate(model_sgd, val_loader, device, is_sora=True)
    avg_rank = model_sgd.get_avg_effective_rank()
    all_ranks_sgd = model_sgd.get_effective_ranks()
    avg_sparsity = np.mean([s['gate_sparsity'] for s in all_ranks_sgd.values()])
    
    sgd_results['train_losses'].append(epoch_loss / num_batches)
    sgd_results['val_losses'].append(val_metrics['loss'])
    sgd_results['val_mccs'].append(val_metrics['mcc'])
    sgd_results['val_accs'].append(val_metrics['accuracy'])
    sgd_results['epoch_times'].append(epoch_time)
    sgd_results['effective_rank_history'].append(avg_rank)
    sgd_results['gate_sparsity_history'].append(avg_sparsity)
    
    if val_metrics['mcc'] > sgd_results['best_mcc']:
        sgd_results['best_mcc'] = val_metrics['mcc']
        sgd_results['best_epoch'] = epoch + 1
    
    print(f"Epoch {epoch+1}/{config.num_epochs} | {epoch_time:.1f}s | Loss: {epoch_loss/num_batches:.4f} | MCC: {val_metrics['mcc']:.4f} | EffRank: {avg_rank:.1f}/{config.sora_rank} | Sparsity: {avg_sparsity:.2%}")

sgd_results['total_time'] = time.time() - start_time
sgd_results['effective_ranks'] = model_sgd.get_effective_ranks()
sgd_results['param_counts'] = {**count_parameters(model_sgd, 'sora_sgd'), **model_sgd.count_trainable_parameters()}

print("\n" + "=" * 60)
print("SoRA Proximal vs SoRA SGD-Subgradient")
print("=" * 60)
print(f"{'Metric':<25} {'Proximal':>12} {'SGD Subgrad':>12}")
print("-" * 50)
print(f"{'Best MCC':<25} {sora_results['best_mcc']:>12.4f} {sgd_results['best_mcc']:>12.4f}")
print(f"{'Final Avg Eff. Rank':<25} {sora_results['effective_rank_history'][-1]:>12.1f} {sgd_results['effective_rank_history'][-1]:>12.1f}")
print(f"{'Final Sparsity':<25} {sora_results['gate_sparsity_history'][-1]:>11.2%} {sgd_results['gate_sparsity_history'][-1]:>11.2%}")
print(f"{'Total Time':<25} {sora_results['total_time']:>11.1f}s {sgd_results['total_time']:>11.1f}s")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('SoRA: Proximal vs SGD Subgradient on CoLA', fontsize=14, fontweight='bold')

axes[0].plot(sora_results['val_mccs'], '-o', label='Proximal', color='#4CAF50', markersize=4)
axes[0].plot(sgd_results['val_mccs'], '-o', label='SGD Subgrad', color='#F44336', markersize=4)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MCC'); axes[0].set_title('Validation MCC')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(sora_results['effective_rank_history'], '-o', label='Proximal', color='#4CAF50', markersize=4)
axes[1].plot(sgd_results['effective_rank_history'], '-o', label='SGD Subgrad', color='#F44336', markersize=4)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Avg Effective Rank'); axes[1].set_title('Effective Rank')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(sora_results['gate_sparsity_history'], '-o', label='Proximal', color='#4CAF50', markersize=4)
axes[2].plot(sgd_results['gate_sparsity_history'], '-o', label='SGD Subgrad', color='#F44336', markersize=4)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Sparsity'); axes[2].set_title('Gate Sparsity')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'sora_prox_vs_sgd_cola.png'), dpi=150, bbox_inches='tight')
plt.show()

Training SoRA with SGD Subgradient (instead of Proximal)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

Injected SoRA adapters into 36 layers
Epoch 1/10 | 177.3s | Loss: 0.6338 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 2/10 | 176.8s | Loss: 0.6109 | MCC: 0.0000 | EffRank: 4.9/8 | Sparsity: 38.54%
Epoch 3/10 | 176.6s | Loss: 0.6083 | MCC: 0.0000 | EffRank: 5.4/8 | Sparsity: 32.99%
Epoch 4/10 | 176.9s | Loss: 0.6112 | MCC: 0.0000 | EffRank: 5.5/8 | Sparsity: 31.60%
Epoch 5/10 | 176.8s | Loss: 0.6100 | MCC: 0.0000 | EffRank: 5.9/8 | Sparsity: 25.69%
Epoch 6/10 | 177.0s | Loss: 0.6087 | MCC: 0.0000 | EffRank: 5.9/8 | Sparsity: 26.74%
Epoch 7/10 | 176.6s | Loss: 0.6071 | MCC: 0.0000 | EffRank: 6.2/8 | Sparsity: 21.88%
Epoch 9/10 | 176.2s | Loss: 0.6078 | MCC: 0.0000 | EffRank: 6.2/8 | Sparsity: 22.22%
Epoch 10/10 | 176.6s | Loss: 0.6062 | MCC: 0.0000 | EffRank: 6.5/8 | Sparsity: 18.40%

SoRA Proximal vs SoRA SGD-Subgradient
Metric                        Proximal  SGD Subgrad
--------------------------------------------------
Best MCC                        0.3711       0.0000
Fin

In [20]:
#  SGDSubgradientOptimizer  — define this alongside ProximalSGD

class SGDSubgradientOptimizer:
    """
    SGD with L1 subgradient for gate vectors.

    Update rule (per step, using gradient already in param.grad):
        g ← g - lr * (∇L_task + λ·sign(g))

    IMPORTANT: param.grad must contain ONLY ∇L_task when step() is called.
    Do NOT include the L1 term in the backward loss — it is added here.
    """

    def __init__(self, gate_params, lr=1e-2, lam=0.0):
        self.gate_params = list(gate_params)
        self.lr  = lr
        self.lam = lam

    def step(self):
        for param in self.gate_params:
            with torch.no_grad():
                if param.grad is not None:
                    subgrad_l1 = torch.sign(param.data)
                    total_grad = param.grad + self.lam * subgrad_l1
                    param.data.sub_(self.lr * total_grad)

    def zero_grad(self):
        for param in self.gate_params:
            if param.grad is not None:
                param.grad.zero_()


import gc; gc.collect(); torch.cuda.empty_cache()

set_seed(config.seed)
print("=" * 70)
print("Training SoRA with SGD Subgradient (instead of Proximal)")
print("=" * 70)

base_model_sgd = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS).float()

model_sgd = SoRAModel(
    base_model_sgd, config.target_modules,
    rank=config.sora_rank, alpha=config.lora_alpha,
    dropout=config.lora_dropout).to(device)

for n, p in model_sgd.base_model.named_parameters():
    if 'classifier' in n or 'pooler' in n:
        p.requires_grad = True

sora_params_sgd = model_sgd.get_sora_parameters()
gate_params_sgd = list(model_sgd.get_gate_parameters())
head_params_sgd = [p for n, p in model_sgd.base_model.named_parameters()
                   if p.requires_grad and ('classifier' in n or 'pooler' in n)]

optimizer_ab_sgd = AdamW([
    {'params': sora_params_sgd, 'lr': 3e-4},
    {'params': head_params_sgd, 'lr': 2e-3},
], weight_decay=config.weight_decay)

sgd_subgrad_optimizer = SGDSubgradientOptimizer(
    gate_params_sgd, lr=config.sora_proximal_lr, lam=0.0)

total_steps  = len(train_loader) * config.num_epochs
warmup_steps = int(total_steps * config.warmup_ratio)
scheduler_sgd = torch.optim.lr_scheduler.LambdaLR(
    optimizer_ab_sgd,
    lr_lambda=lambda step: min(1.0, step / max(warmup_steps, 1)))

sgd_results = {
    'method': 'sora_sgd',
    'train_losses': [], 'val_losses': [], 'val_mccs': [], 'val_accs': [],
    'epoch_times': [], 'best_mcc': -1.0, 'best_epoch': 0,
    'effective_rank_history': [], 'gate_sparsity_history': [],
}
start_time = time.time()

LEARN_EPOCHS = config.num_epochs // 2  

for epoch in range(config.num_epochs):
    epoch_start = time.time()
    model_sgd.train()
    epoch_loss, num_batches = 0.0, 0

    if epoch < LEARN_EPOCHS:
        sgd_subgrad_optimizer.lam = 0.0
        phase = "LEARN"
    else:
        ramp = min(1.0, (epoch - LEARN_EPOCHS + 1) / max(LEARN_EPOCHS // 2, 1))
        sgd_subgrad_optimizer.lam = config.sora_lambda * ramp
        phase = "PRUNE"

    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model_sgd(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels'],
        )
        task_loss = outputs.loss
        if torch.isnan(task_loss):
            optimizer_ab_sgd.zero_grad()
            sgd_subgrad_optimizer.zero_grad()
            continue

        task_loss.backward()

        torch.nn.utils.clip_grad_norm_(sora_params_sgd, config.max_grad_norm)

        optimizer_ab_sgd.step()
        scheduler_sgd.step()


        sgd_subgrad_optimizer.step()

        optimizer_ab_sgd.zero_grad()
        sgd_subgrad_optimizer.zero_grad()

        epoch_loss += task_loss.item()
        num_batches += 1

    epoch_time   = time.time() - epoch_start
    val_metrics  = evaluate(model_sgd, val_loader, device, is_sora=True)
    avg_rank     = model_sgd.get_avg_effective_rank()
    all_ranks    = model_sgd.get_effective_ranks()
    avg_sparsity = np.mean([s['gate_sparsity'] for s in all_ranks.values()])

    sgd_results['train_losses'].append(epoch_loss / num_batches)
    sgd_results['val_losses'].append(val_metrics['loss'])
    sgd_results['val_mccs'].append(val_metrics['mcc'])
    sgd_results['val_accs'].append(val_metrics['accuracy'])
    sgd_results['epoch_times'].append(epoch_time)
    sgd_results['effective_rank_history'].append(avg_rank)
    sgd_results['gate_sparsity_history'].append(avg_sparsity)

    if val_metrics['mcc'] > sgd_results['best_mcc']:
        sgd_results['best_mcc']   = val_metrics['mcc']
        sgd_results['best_epoch'] = epoch + 1

    print(
        f"Epoch {epoch+1}/{config.num_epochs} [{phase}] | {epoch_time:.1f}s | "
        f"Loss: {epoch_loss/num_batches:.4f} | "
        f"λ: {sgd_subgrad_optimizer.lam:.4f} | "
        f"MCC: {val_metrics['mcc']:.4f} | "
        f"EffRank: {avg_rank:.1f}/{config.sora_rank} | "
        f"Sparsity: {avg_sparsity:.2%}"
    )

sgd_results['total_time']      = time.time() - start_time
sgd_results['effective_ranks'] = model_sgd.get_effective_ranks()
sgd_results['param_counts']    = model_sgd.count_trainable_parameters()

print("\n" + "=" * 60)
print("SoRA Proximal vs SoRA SGD-Subgradient")
print("=" * 60)
print(f"{'Metric':<25} {'Proximal':>12} {'SGD Subgrad':>12}")
print("-" * 50)
print(f"{'Best MCC':<25} {sora_results['best_mcc']:>12.4f} {sgd_results['best_mcc']:>12.4f}")
print(f"{'Final Avg Eff. Rank':<25} {sora_results['effective_rank_history'][-1]:>12.1f} {sgd_results['effective_rank_history'][-1]:>12.1f}")
print(f"{'Final Sparsity':<25} {sora_results['gate_sparsity_history'][-1]:>11.2%} {sgd_results['gate_sparsity_history'][-1]:>11.2%}")
print(f"{'Total Time':<25} {sora_results['total_time']:>11.1f}s {sgd_results['total_time']:>11.1f}s")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('SoRA: Proximal vs SGD Subgradient on CoLA',
             fontsize=14, fontweight='bold')

axes[0].plot(sora_results['val_mccs'], '-o', label='Proximal',
             color='#4CAF50', markersize=4)
axes[0].plot(sgd_results['val_mccs'],  '-o', label='SGD Subgrad',
             color='#F44336', markersize=4)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MCC')
axes[0].set_title('Validation MCC')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(sora_results['effective_rank_history'], '-o', label='Proximal',
             color='#4CAF50', markersize=4)
axes[1].plot(sgd_results['effective_rank_history'],  '-o', label='SGD Subgrad',
             color='#F44336', markersize=4)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Avg Effective Rank')
axes[1].set_title('Effective Rank')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(sora_results['gate_sparsity_history'], '-o', label='Proximal',
             color='#4CAF50', markersize=4)
axes[2].plot(sgd_results['gate_sparsity_history'],  '-o', label='SGD Subgrad',
             color='#F44336', markersize=4)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Sparsity')
axes[2].set_title('Gate Sparsity')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'sora_prox_vs_sgd_cola.png'),
            dpi=150, bbox_inches='tight')
plt.show()

Training SoRA with SGD Subgradient (instead of Proximal)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

Injected SoRA adapters into 36 layers
Epoch 1/10 [LEARN] | 178.5s | Loss: 0.6250 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 2/10 [LEARN] | 176.1s | Loss: 0.6187 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 3/10 [LEARN] | 176.3s | Loss: 0.6133 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 4/10 [LEARN] | 175.6s | Loss: 0.6127 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 5/10 [LEARN] | 175.5s | Loss: 0.6131 | λ: 0.0000 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 6/10 [PRUNE] | 176.2s | Loss: 0.6109 | λ: 0.2500 | MCC: 0.0000 | EffRank: 8.0/8 | Sparsity: 0.00%
Epoch 7/10 [PRUNE] | 176.0s | Loss: 0.6154 | λ: 0.5000 | MCC: 0.0000 | EffRank: 7.5/8 | Sparsity: 6.25%
Epoch 8/10 [PRUNE] | 175.9s | Loss: 0.6114 | λ: 0.5000 | MCC: 0.0000 | EffRank: 7.5/8 | Sparsity: 6.60%
Epoch 9/10 [PRUNE] | 176.5s | Loss: 0.6085 | λ: 0.5000 | MCC: 0.0000 | EffRank: 7.6/8 | Sparsity: 5.56%
Epoch 10/10 [PRUNE] | 176.

In [42]:
import numpy as np

def sgd_subgradient_update_numpy(g, grad_task, lr, lam, sign_at_zero='zero'):
    """
    SGD with L1 subgradient.
    g:           gate vector (shape [r])
    grad_task:   ∂L_0/∂g (shape [r])
    lr:          learning rate
    lam:         L1 penalty strength
    sign_at_zero: 'zero', 'plus', or 'random'
    """
    
    subgrad = np.sign(g)  
    
    if sign_at_zero == 'plus':
        subgrad[g == 0] = 1.0
    elif sign_at_zero == 'random':
        zeros = (g == 0)
        subgrad[zeros] = np.random.choice([-1.0, 1.0], size=zeros.sum())
    
    
    total_grad = grad_task + lam * subgrad
    g_new = g - lr * total_grad
    return g_new

def proximal_update_numpy(g, grad_task, lr, lam):
    
    g_half = g - lr * grad_task
   
    g_new = np.sign(g_half) * np.maximum(np.abs(g_half) - lr * lam, 0.0)
    return g_new


np.random.seed(42)
r = 8
g_sgd  = np.ones(r)
g_prox = np.ones(r)
lr, lam, n_steps = 0.01, 1.0, 500

sgd_history, prox_history = [], []

for t in range(n_steps):
    
    grad_task = np.array([0.1, 0.08, 0.12, 0.09, 0.001, 0.002, 0.0005, 0.001])
    grad_task += np.random.randn(r) * 0.01  # noise
    
    g_sgd  = sgd_subgradient_update_numpy(g_sgd, grad_task, lr, lam)
    g_prox = proximal_update_numpy(g_prox, grad_task, lr, lam)
    
    sgd_history.append(g_sgd.copy())
    prox_history.append(g_prox.copy())

sgd_history  = np.array(sgd_history)
prox_history = np.array(prox_history)

print("Final gates (SGD subgradient):")
print(f"  Values: {g_sgd.round(6)}")
print(f"  Exact zeros: {(g_sgd == 0.0).sum()}/{r}")
print(f"  Near zeros (<1e-3): {(np.abs(g_sgd) < 1e-3).sum()}/{r}")

print("\nFinal gates (Proximal):")
print(f"  Values: {g_prox.round(6)}")
print(f"  Exact zeros: {(g_prox == 0.0).sum()}/{r}")
print(f"  Near zeros (<1e-3): {(np.abs(g_prox) < 1e-3).sum()}/{r}")

Final gates (SGD subgradient):
  Values: [ 0.000348 -0.001848  0.002498  0.007803 -0.00841   0.009365 -0.004173
 -0.005812]
  Exact zeros: 0/8
  Near zeros (<1e-3): 1/8

Final gates (Proximal):
  Values: [-0. -0. -0. -0.  0.  0. -0. -0.]
  Exact zeros: 8/8
  Near zeros (<1e-3): 8/8


In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SGDSubgradientOptimizer:
    """SGD with explicit L1 subgradient for gate vectors."""
    def __init__(self, gate_params, lr=1e-2, lam=0.0, sign_at_zero='zero'):
        self.gate_params = list(gate_params)
        self.lr = lr
        self.lam = lam
        self.sign_at_zero = sign_at_zero

    def step(self):
        for param in self.gate_params:
            with torch.no_grad():
                if param.grad is not None:
                    subgrad = torch.sign(param.data)
                    if self.sign_at_zero == 'plus':
                        subgrad[param.data == 0] = 1.0
                    elif self.sign_at_zero == 'random':
                        zeros = (param.data == 0)
                        subgrad[zeros] = torch.randint(0, 2, (zeros.sum(),),
                            device=param.device).float() * 2 - 1
                    total_grad = param.grad + self.lam * subgrad
                    param.data.sub_(self.lr * total_grad)

    def zero_grad(self):
        for param in self.gate_params:
            if param.grad is not None:
                param.grad.zero_()


class ProximalSGD:
    """Proximal gradient descent with soft-thresholding for L1."""
    def __init__(self, gate_params, lr=1e-2, lam=0.0):
        self.gate_params = list(gate_params)
        self.lr = lr
        self.lam = lam

    @staticmethod
    def soft_threshold(x, threshold):
        return torch.sign(x) * torch.clamp(x.abs() - threshold, min=0.0)

    def step(self):
        threshold = self.lr * self.lam
        for param in self.gate_params:
            with torch.no_grad():
                if param.grad is not None:
                    param.data.sub_(self.lr * param.grad)  # gradient step
                param.data = self.soft_threshold(param.data, threshold)  # proximal step

    def zero_grad(self):
        for param in self.gate_params:
            if param.grad is not None:
                param.grad.zero_()


torch.manual_seed(42)
r, d_in, d_out = 8, 32, 32
lr, lam, n_steps = 0.01, 1.0, 500

sora_sgd  = SoRALinear(d_in, d_out, rank=r, alpha=r, dropout=0.0)
sora_prox = SoRALinear(d_in, d_out, rank=r, alpha=r, dropout=0.0)

sora_prox.lora_A.data.copy_(sora_sgd.lora_A.data)
sora_prox.lora_B.data.copy_(sora_sgd.lora_B.data)
sora_prox.gate.data.copy_(sora_sgd.gate.data)

ab_opt_sgd  = torch.optim.SGD([sora_sgd.lora_A, sora_sgd.lora_B], lr=lr)
ab_opt_prox = torch.optim.SGD([sora_prox.lora_A, sora_prox.lora_B], lr=lr)
sgd_opt  = SGDSubgradientOptimizer([sora_sgd.gate], lr=lr, lam=lam)
prox_opt = ProximalSGD([sora_prox.gate], lr=lr, lam=lam)

x = torch.randn(16, d_in)
target = torch.randn(16, d_out)

sgd_gates, prox_gates = [], []

for step in range(n_steps):
    out_s = sora_sgd(x)
    loss_s = F.mse_loss(out_s, target)
    loss_s.backward()
    ab_opt_sgd.step(); ab_opt_sgd.zero_grad()
    sgd_opt.step(); sgd_opt.zero_grad()
    
    out_p = sora_prox(x)
    loss_p = F.mse_loss(out_p, target)
    loss_p.backward()
    ab_opt_prox.step(); ab_opt_prox.zero_grad()
    prox_opt.step(); prox_opt.zero_grad()
    
    sgd_gates.append(sora_sgd.gate.detach().clone().numpy())
    prox_gates.append(sora_prox.gate.detach().clone().numpy())

sgd_gates  = np.array(sgd_gates)
prox_gates = np.array(prox_gates)

print("PyTorch Results:")
print(f"  SGD  exact zeros: {(sora_sgd.gate.data.abs() < 1e-8).sum()}/{r}")
print(f"  Prox exact zeros: {(sora_prox.gate.data.abs() < 1e-8).sum()}/{r}")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i in range(r):
    axes[0].plot(sgd_gates[:, i], alpha=0.7, label=f'g_{i}')
    axes[1].plot(prox_gates[:, i], alpha=0.7, label=f'g_{i}')
axes[0].set_title('SGD Subgradient'); axes[0].axhline(0, color='k', ls='--', lw=0.5)
axes[1].set_title('Proximal (Soft-threshold)'); axes[1].axhline(0, color='k', ls='--', lw=0.5)
for ax in axes:
    ax.set_xlabel('Step'); ax.set_ylabel('Gate value'); ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('sgd_vs_proximal_gates.png', dpi=150); plt.show()

PyTorch Results:
  SGD  exact zeros: 0/8
  Prox exact zeros: 8/8


## Part 2b: Mathematical Comparison

### Proximal Soft-Thresholding Update

The proximal gradient descent update for the L1-regularized objective is:

$$g^{(t+1)} = \text{prox}_{\eta\lambda\|\cdot\|_1}\left(g^{(t)} - \eta \nabla_{g} L_0\right) = S_{\eta\lambda}\left(g^{(t)} - \eta \nabla_{g} L_0\right)$$

where the soft-thresholding operator is:

$$S_\xi(v)_i = \text{sign}(v_i) \max(|v_i| - \xi, 0)$$

**Key property**: Any component with $|g_i^{(t)} - \eta \nabla_{g_i} L_0| < \eta\lambda$ is driven **exactly to zero**.

### SGD Subgradient Update

$$g^{(t+1)} = g^{(t)} - \eta\left(\nabla_g L_0 + \lambda \cdot \partial\|g\|_1\right)$$

where $\partial\|g\|_1$ is a subgradient of the L1 norm:
$$(\partial\|g\|_1)_i = \begin{cases} \text{sign}(g_i) & g_i \neq 0 \\ s_i \in [-1, 1] & g_i = 0\end{cases}$$

**Key property**: SGD subgradient **oscillates around zero** but rarely produces exact zeros because:
- When $g_i > 0$: update pushes $g_i$ toward $g_i - \eta\lambda$ (subtracts)
- When $g_i < 0$: update pushes $g_i$ toward $g_i + \eta\lambda$ (adds)
- This causes oscillation around 0 without settling exactly at 0

### Why Proximal is Better for Sparsity

The proximal operator solves:
$$\text{prox}_{\eta\lambda\|\cdot\|_1}(v) = \arg\min_u \frac{1}{2}\|u - v\|^2 + \eta\lambda\|u\|_1$$

This is a **two-step** process: gradient step then projection. The projection (soft-thresholding) explicitly checks "is this component close enough to zero?" and if so, **sets it exactly to zero**.

SGD treats $\lambda\|g\|_1$ as just another term in the loss and computes a subgradient. It never explicitly "projects" onto the sparse set. The gradient of $|g_i|$ at $g_i = 0$ is **undefined** (we pick a subgradient from $[-1, 1]$), so the iterates hover around zero.

## Part 2c: Does the Choice of Subgradient Matter?

At $g_i = 0$, the subdifferential is $\partial|g_i| = [-1, 1]$, and we must choose a specific value $s_i \in [-1, 1]$.

**Common choices:**
1. $s_i = 0$ (what `np.sign(0)` gives) — the update ignores L1 at zero, so $g_i$ moves only due to task loss. If the task loss gradient is nonzero, $g_i$ moves away from zero.
2. $s_i = \text{sign}(\nabla_{g_i} L_0)$ — tries to counteract the task gradient, but can still overshoot.
3. Any $s_i \in [-1, 1]$ — the convergence guarantee (rate $O(1/\sqrt{t})$) holds for **any** valid subgradient choice.

**Theoretical impact**: The specific subgradient choice affects the trajectory but NOT the convergence rate. All valid choices converge at $O(1/\sqrt{t})$, which is **slower** than proximal gradient descent's $O(1/t)$ rate. No subgradient choice can match the sparsity-inducing property of the proximal operator.

**Conclusion**: The proximal method is strictly superior for inducing sparsity. The subgradient choice in SGD is a secondary concern — the fundamental limitation is the absence of an explicit projection step.

In [2]:
!pip install -q mamba-ssm --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 44.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 67.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 24.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 50.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 41.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatibl

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SoRALinearV2(nn.Module):
    """Enhanced SoRA for recurrent architectures — identical math, cleaner API."""
    def __init__(self, in_features, out_features, rank=8, alpha=16.0, dropout=0.1):
        super().__init__()
        self.rank = rank
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.empty(out_features, rank))
        self.gate = nn.Parameter(torch.ones(rank))
        self.dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
    
    def forward(self, x):
        return F.linear(self.dropout(F.linear(x, self.lora_A)) * self.gate, self.lora_B) * self.scaling
    
    def get_effective_rank(self, threshold=1e-5):
        return (self.gate.abs() > threshold).sum().item()


class RecurrentSoRAWrapper(nn.Module):

    def __init__(self, base_model, target_module_names, rank=8, alpha=16.0, dropout=0.1,
                 task_type='classification', num_labels=2, hidden_size=768):
        super().__init__()
        self.base_model = base_model
        self.sora_layers = {}
        self.task_type = task_type
        
        for param in self.base_model.parameters():
            param.requires_grad = False
        
        # Inject SoRA
        injected = 0
        for name, module in self.base_model.named_modules():
            if any(t in name for t in target_module_names) and isinstance(module, nn.Linear):
                sora = SoRALinearV2(module.in_features, module.out_features, rank, alpha, dropout)
                key = name.replace('.', '_')
                self.sora_layers[key] = sora
                self.add_module(f"sora_{key}", sora)
                
                def make_hook(orig_fn, s):
                    def hook(x):
                        return orig_fn(x) + s(x)
                    return hook
                module.forward = make_hook(module.forward, sora)
                injected += 1
        
        if task_type == 'classification':
            self.classifier = nn.Sequential(
                nn.Linear(hidden_size, hidden_size // 2),
                nn.GELU(),
                nn.Dropout(0.1),
                nn.Linear(hidden_size // 2, num_labels),
            )
        
        print(f"Injected SoRA into {injected} layers")
    
    def get_sora_parameters(self):
        return [p for n, p in self.named_parameters() if 'sora_' in n and 'gate' not in n]
    
    def get_gate_parameters(self):
        return [p for n, p in self.named_parameters() if 'gate' in n and 'sora_' in n]
    
    def get_classifier_parameters(self):
        return list(self.classifier.parameters()) if hasattr(self, 'classifier') else []
    
    def get_effective_ranks(self):
        return {k: {'effective_rank': v.get_effective_rank(), 'max_rank': v.rank,
                     'gate_values': v.gate.detach().cpu().tolist()} for k, v in self.sora_layers.items()}
    
    def get_avg_effective_rank(self):
        ranks = [v.get_effective_rank() for v in self.sora_layers.values()]
        return sum(ranks) / len(ranks) if ranks else 0.0

try:
    from mamba_ssm import Mamba
    
    class MambaClassifier(nn.Module):
        """Simple Mamba-based sequence classifier."""
        def __init__(self, vocab_size, d_model=256, n_layers=4, d_state=16, d_conv=4, expand=2, num_labels=2):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, d_model)
            self.layers = nn.ModuleList([
                Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
                for _ in range(n_layers)
            ])
            self.norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, num_labels)
        
        def forward(self, input_ids, attention_mask=None, labels=None):
            x = self.embedding(input_ids)
            for layer in self.layers:
                x = layer(x) + x  # residual
            x = self.norm(x)
            # Pool: mean over sequence (use attention_mask if available)
            if attention_mask is not None:
                mask = attention_mask.unsqueeze(-1).float()
                x = (x * mask).sum(dim=1) / mask.sum(dim=1)
            else:
                x = x.mean(dim=1)
            logits = self.head(x)
            
            loss = None
            if labels is not None:
                loss = F.cross_entropy(logits, labels)
            
            from types import SimpleNamespace
            return SimpleNamespace(loss=loss, logits=logits)
    
    MAMBA_AVAILABLE = True
    print("✓ Mamba available!")
    
    # --- Mamba SoRA target modules ---
    # Mamba's key projections: in_proj, out_proj, x_proj, dt_proj
    MAMBA_TARGET_MODULES = ["in_proj", "out_proj", "x_proj", "dt_proj"]
    
except ImportError:
    MAMBA_AVAILABLE = False
    print("✗ Mamba not available (needs CUDA). Will use a lightweight SSM fallback.")

# Fallback: Simple SSM-like model if mamba-ssm doesn't install

class SimpleSSMLayer(nn.Module):
    """Simplified State Space Model layer (S4-style diagonal)."""
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        
        # SSM parameters: A (diagonal), B, C, D
        self.A_log = nn.Parameter(torch.log(torch.rand(d_model, d_state) * 0.9 + 0.1))
        self.B_proj = nn.Linear(d_model, d_state)
        self.C_proj = nn.Linear(d_state, d_model)
        self.D = nn.Parameter(torch.ones(d_model))
        
        # Input/output projections
        self.in_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        residual = x
        x = self.norm(x)
        x = self.in_proj(x)
        
        B, L, D = x.shape
        A = -torch.exp(self.A_log.clamp(max=3.0))
        
        # Discretize and scan (simplified)
        B_input = self.B_proj(x)  # (B, L, d_state)
        
        h = torch.zeros(B, self.d_model, self.d_state, device=x.device)
        outputs = []
        for t in range(L):
            h = h * torch.exp(A.unsqueeze(0)) + B_input[:, t, :].unsqueeze(1).expand_as(h)
            y_t = (h * self.C_proj.weight.T.unsqueeze(0)).sum(-1) + x[:, t] * self.D
            outputs.append(y_t)
        
        y = torch.stack(outputs, dim=1)
        y = self.out_proj(y)
        return y + residual


class SimpleSSMClassifier(nn.Module):
    """SSM-based classifier for CoLA."""
    def __init__(self, vocab_size, d_model=256, n_layers=4, d_state=16, num_labels=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([SimpleSSMLayer(d_model, d_state) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_labels)
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            x = x.mean(dim=1)
        logits = self.head(x)
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        from types import SimpleNamespace
        return SimpleNamespace(loss=loss, logits=logits)

SSM_TARGET_MODULES = ["in_proj", "out_proj", "B_proj", "C_proj"]

# Similarly: Simple xLSTM-like model

class SimplexLSTMCell(nn.Module):
    """Simplified xLSTM cell (sLSTM variant with exponential gating)."""
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # Gates
        self.i_gate = nn.Linear(d_model * 2, d_model)
        self.f_gate = nn.Linear(d_model * 2, d_model)
        self.o_gate = nn.Linear(d_model * 2, d_model)
        self.z_gate = nn.Linear(d_model * 2, d_model)
    
    def forward(self, x_t, h_prev, c_prev):
        combined = torch.cat([x_t, h_prev], dim=-1)
        i = torch.exp(self.i_gate(combined))  # exponential gating (xLSTM key innovation)
        f = torch.sigmoid(self.f_gate(combined))
        o = torch.sigmoid(self.o_gate(combined))
        z = torch.tanh(self.z_gate(combined))
        
        c = f * c_prev + i * z
        h = o * torch.tanh(c)
        return h, c


class SimplexLSTMLayer(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.cell = SimplexLSTMCell(d_model)
        self.in_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        residual = x
        x = self.norm(x)
        x = self.in_proj(x)
        B, L, D = x.shape
        h = torch.zeros(B, D, device=x.device)
        c = torch.zeros(B, D, device=x.device)
        outputs = []
        for t in range(L):
            h, c = self.cell(x[:, t], h, c)
            outputs.append(h)
        y = torch.stack(outputs, dim=1)
        return self.out_proj(y) + residual


class SimplexLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=4, num_labels=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([SimplexLSTMLayer(d_model) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_labels)
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            x = x.mean(dim=1)
        logits = self.head(x)
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        from types import SimpleNamespace
        return SimpleNamespace(loss=loss, logits=logits)

XLSTM_TARGET_MODULES = ["i_gate", "f_gate", "o_gate", "z_gate", "in_proj", "out_proj"]

print("✓ Model definitions ready")

✓ Mamba available!
✓ Model definitions ready


In [9]:
set_seed(config.seed)
vocab_size = tokenizer.vocab_size
d_model = 256
rank = 8

# xLSTM + SoRA 
print("=" * 70)
print("Training xLSTM + SoRA")
print("=" * 70)

xlstm_base = SimplexLSTMClassifier(vocab_size, d_model=d_model, n_layers=4, num_labels=2)

# First, pretrain the base model briefly since it's from scratch
print("Pre-training xLSTM base (3 epochs)...")
xlstm_base = xlstm_base.to(device)
opt_pretrain = AdamW(xlstm_base.parameters(), lr=1e-3, weight_decay=0.01)

for epoch in range(3):
    xlstm_base.train()
    total_loss, n = 0, 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = xlstm_base(**batch)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(xlstm_base.parameters(), 1.0)
        opt_pretrain.step(); opt_pretrain.zero_grad()
        total_loss += out.loss.item(); n += 1
    val = evaluate(xlstm_base, val_loader, device)
    print(f"  Pretrain epoch {epoch+1}: loss={total_loss/n:.4f}, val_mcc={val['mcc']:.4f}")

# Now apply SoRA
xlstm_sora = RecurrentSoRAWrapper(
    xlstm_base, XLSTM_TARGET_MODULES, rank=rank, alpha=16.0, dropout=0.1,
    task_type='classification', num_labels=2, hidden_size=d_model
).to(device)

for n, p in xlstm_base.named_parameters():
    if 'head' in n:
        p.requires_grad = True

sora_p = xlstm_sora.get_sora_parameters()
gate_p = xlstm_sora.get_gate_parameters()
cls_p = [p for n, p in xlstm_base.named_parameters() if p.requires_grad and 'head' in n]

opt_xlstm = AdamW(sora_p + cls_p, lr=2e-4, weight_decay=0.01)
prox_xlstm = ProximalSGD(gate_p, lr=2e-4, lam=0.01)
total_steps_xlstm = len(train_loader) * config.num_epochs
sched_xlstm = torch.optim.lr_scheduler.OneCycleLR(opt_xlstm, max_lr=2e-4, total_steps=total_steps_xlstm, pct_start=0.06)

xlstm_results = {'method': 'xlstm_sora', 'val_mccs': [], 'effective_rank_history': [], 'gate_sparsity_history': [], 'epoch_times': []}
start = time.time()

for epoch in range(config.num_epochs):
    es = time.time()
    xlstm_sora.base_model.train()
    xlstm_sora.train()
    total_loss, n = 0, 0
    lam_scale = min(1.0, (epoch + 1) / (config.num_epochs * 0.5))
    prox_xlstm.lam = 0.01 * lam_scale
    
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = xlstm_base(**batch)       # was: xlstm_base(**batch)
        l1 = sum(s.gate.abs().sum() for s in xlstm_sora.sora_layers.values()) * prox_xlstm.lam
        loss = out.loss + l1
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sora_p, 1.0)
        opt_xlstm.step(); sched_xlstm.step()
        prox_xlstm.step()
        opt_xlstm.zero_grad()           # zero_grad AFTER proximal step
        prox_xlstm.zero_grad()
        total_loss += out.loss.item(); n += 1
    
    val = evaluate(xlstm_base, val_loader, device)  # was: xlstm_base
    avg_rank = xlstm_sora.get_avg_effective_rank()
    ranks = xlstm_sora.get_effective_ranks()
    
    xlstm_results['val_mccs'].append(val['mcc'])
    xlstm_results['effective_rank_history'].append(avg_rank)
    xlstm_results['epoch_times'].append(time.time() - es)
    
    print(f"Epoch {epoch+1} | Loss: {total_loss/n:.4f} | MCC: {val['mcc']:.4f} | EffRank: {avg_rank:.1f}/{rank}")

xlstm_results['total_time'] = time.time() - start
xlstm_results['best_mcc'] = max(xlstm_results['val_mccs'])
print(f"\nxLSTM+SoRA Best MCC: {xlstm_results['best_mcc']:.4f}")

NameError: name 'set_seed' is not defined

In [33]:
set_seed(config.seed)

# SSM + SoRA
print("=" * 70)
print("Training SSM (Mamba-like) + SoRA")
print("=" * 70)

if MAMBA_AVAILABLE:
    ssm_base = MambaClassifier(vocab_size, d_model=d_model, n_layers=4, num_labels=2)
    target_mods = MAMBA_TARGET_MODULES
else:
    ssm_base = SimpleSSMClassifier(vocab_size, d_model=d_model, n_layers=4, num_labels=2)
    target_mods = SSM_TARGET_MODULES

ssm_base = ssm_base.to(device)

# Pretrain
print("Pre-training SSM base (3 epochs)...")
opt_pre = AdamW(ssm_base.parameters(), lr=5e-4, weight_decay=0.01)
for epoch in range(3):
    ssm_base.train()
    total_loss, n = 0, 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = ssm_base(**batch)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(ssm_base.parameters(), 1.0)
        opt_pre.step(); opt_pre.zero_grad()
        total_loss += out.loss.item(); n += 1
    val = evaluate(ssm_base, val_loader, device)
    print(f"  Pretrain epoch {epoch+1}: loss={total_loss/n:.4f}, val_mcc={val['mcc']:.4f}")

# Apply SoRA
ssm_sora = RecurrentSoRAWrapper(
    ssm_base, target_mods, rank=rank, alpha=16.0, dropout=0.1,
    task_type='classification', num_labels=2, hidden_size=d_model
).to(device)

for n_p, p in ssm_base.named_parameters():
    if 'head' in n_p:
        p.requires_grad = True

sora_p2 = ssm_sora.get_sora_parameters()
gate_p2 = ssm_sora.get_gate_parameters()
cls_p2 = [p for n_p, p in ssm_base.named_parameters() if p.requires_grad and 'head' in n_p]

opt_ssm = AdamW(sora_p2 + cls_p2, lr=2e-4, weight_decay=0.01)
prox_ssm = ProximalSGD(gate_p2, lr=2e-4, lam=0.01)
total_steps_ssm = len(train_loader) * config.num_epochs
sched_ssm = torch.optim.lr_scheduler.OneCycleLR(opt_ssm, max_lr=2e-4, total_steps=total_steps_ssm, pct_start=0.06)

ssm_results = {'method': 'ssm_sora', 'val_mccs': [], 'effective_rank_history': [], 'epoch_times': []}
start = time.time()

for epoch in range(config.num_epochs):
    es = time.time()
    ssm_base.train(); ssm_sora.train()
    total_loss, n = 0, 0
    lam_scale = min(1.0, (epoch + 1) / (config.num_epochs * 0.5))
    prox_ssm.lam = 0.01 * lam_scale
    
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = ssm_base(**batch)
        l1 = sum(s.gate.abs().sum() for s in ssm_sora.sora_layers.values()) * prox_ssm.lam
        loss = out.loss + l1
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sora_p2, 1.0)
        opt_ssm.step(); sched_ssm.step(); opt_ssm.zero_grad()
        prox_ssm.step(); prox_ssm.zero_grad()
        total_loss += out.loss.item(); n += 1
    
    val = evaluate(ssm_base, val_loader, device)
    avg_rank = ssm_sora.get_avg_effective_rank()
    
    ssm_results['val_mccs'].append(val['mcc'])
    ssm_results['effective_rank_history'].append(avg_rank)
    ssm_results['epoch_times'].append(time.time() - es)
    
    print(f"Epoch {epoch+1} | Loss: {total_loss/n:.4f} | MCC: {val['mcc']:.4f} | EffRank: {avg_rank:.1f}/{rank}")

ssm_results['total_time'] = time.time() - start
ssm_results['best_mcc'] = max(ssm_results['val_mccs'])
print(f"\nSSM+SoRA Best MCC: {ssm_results['best_mcc']:.4f}")

Training SSM (Mamba-like) + SoRA
Pre-training SSM base (3 epochs)...
  Pretrain epoch 1: loss=0.6101, val_mcc=0.0268
  Pretrain epoch 2: loss=0.4948, val_mcc=0.1431
  Pretrain epoch 3: loss=0.2697, val_mcc=0.1214
Injected SoRA into 16 layers
Epoch 1 | Loss: 0.1066 | MCC: 0.1348 | EffRank: 8.0/8
Epoch 2 | Loss: 0.0757 | MCC: 0.1452 | EffRank: 8.0/8
Epoch 3 | Loss: 0.0591 | MCC: 0.1506 | EffRank: 8.0/8
Epoch 4 | Loss: 0.0489 | MCC: 0.1664 | EffRank: 8.0/8
Epoch 5 | Loss: 0.0427 | MCC: 0.1651 | EffRank: 8.0/8
Epoch 6 | Loss: 0.0379 | MCC: 0.1692 | EffRank: 8.0/8
Epoch 7 | Loss: 0.0354 | MCC: 0.1753 | EffRank: 8.0/8
Epoch 8 | Loss: 0.0326 | MCC: 0.1506 | EffRank: 8.0/8
Epoch 9 | Loss: 0.0321 | MCC: 0.1692 | EffRank: 8.0/8
Epoch 10 | Loss: 0.0313 | MCC: 0.1637 | EffRank: 8.0/8

SSM+SoRA Best MCC: 0.1753


In [34]:
# Final comparison: all methods
print("=" * 70)
print("FINAL COMPARISON: ALL METHODS")
print("=" * 70)

print(f"\n{'Method':<20} {'Best MCC':>10} {'Total Time':>12}")
print("-" * 45)
for name, res in [('LoRA', lora_results), ('AdaLoRA', ada_results), 
                    ('SoRA (Proximal)', sora_results), ('SoRA (SGD Sub)', sgd_results),
                    ('xLSTM+SoRA', xlstm_results), ('SSM+SoRA', ssm_results)]:
    print(f"{name:<20} {res['best_mcc']:>10.4f} {res['total_time']:>11.1f}s")

# Plot Part 3 results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Part 3: SoRA on Recurrent Architectures (CoLA)', fontsize=14, fontweight='bold')

# MCC comparison
ax = axes[0]
ax.plot(xlstm_results['val_mccs'], '-o', label='xLSTM+SoRA', color='#E91E63', markersize=4)
ax.plot(ssm_results['val_mccs'], '-o', label='SSM+SoRA', color='#00BCD4', markersize=4)
ax.plot(sora_results['val_mccs'], '--', label='DeBERTa+SoRA (ref)', color='#4CAF50', alpha=0.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('MCC'); ax.set_title('Validation MCC')
ax.legend(); ax.grid(True, alpha=0.3)

# Effective rank
ax = axes[1]
ax.plot(xlstm_results['effective_rank_history'], '-o', label='xLSTM+SoRA', color='#E91E63', markersize=4)
ax.plot(ssm_results['effective_rank_history'], '-o', label='SSM+SoRA', color='#00BCD4', markersize=4)
ax.plot(sora_results['effective_rank_history'], '--', label='DeBERTa+SoRA (ref)', color='#4CAF50', alpha=0.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Avg Effective Rank'); ax.set_title('Effective Rank Evolution')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'part3_recurrent_sora.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ All plots saved to /kaggle/working/plots/")
print("✓ Download from the 'Output' tab after the notebook finishes")

FINAL COMPARISON: ALL METHODS

Method                 Best MCC   Total Time
---------------------------------------------
LoRA                     0.3574      3706.9s
AdaLoRA                  0.3919      3724.5s
SoRA (Proximal)          0.3711      3725.9s
SoRA (SGD Sub)           0.0000      1861.1s
xLSTM+SoRA               0.1198      4381.2s
SSM+SoRA                 0.1753        81.1s

✓ All plots saved to /kaggle/working/plots/
✓ Download from the 'Output' tab after the notebook finishes
